# **Redes Neuronales Artificiales**

Red neuronal

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix

def calculate_metrics(tp, fp, fn, tn):
    accuracy = round((tp + tn) / (tp + fp + fn + tn), 6) if (tp + fp + fn + tn) != 0 else 0
    precision = round(tp / (tp + fp), 6) if (tp + fp) != 0 else 0
    recall = round(tp / (tp + fn), 6) if (tp + fn) != 0 else 0
    f1_score = round(2 * (precision * recall) / (precision + recall), 6) if (precision + recall) != 0 else 0
    specificity = round(tn / (tn + fp), 6) if (tn + fp) != 0 else 1

    return accuracy, precision, recall, f1_score, specificity

def calculate_metrics_for_row(row):
    tp = row['TP']
    fp = row['FP']
    fn = row['FN']
    tn = row['TN']

    accuracy, precision, recall, f1_score, specificity = calculate_metrics(tp, fp, fn, tn)

    return accuracy, precision, recall, f1_score, specificity

# se carga el dataframe
df = pd.read_csv('dataframe.csv')

# Inicializar variable de acumulación de resultados
results = []

# Definimos parámetros
hidden_units = (32) #Número de unidades ocultas
random_state = 42 # Semilla para replicar resultados
alpha = 0.0001  # Ajusta alpha según sea necesario (parámetro de regularización ridge)
n_hours = 24 #número de iteraciones

# Definimos el número de pliegues para la validación cruzada
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)

for i in range(n_hours):
    # print("Se está ejecutando la hora: "+str(i))  -  Mensaje de control
    # Crear una nueva columna basada en el cambio de tiempo
    shift_col_name = "Shifted_Column_" + str(i)
    #El siguiente es para horas
    df[shift_col_name] = df[df.columns[1]].shift(-6 * i)

    # Crear la columna 'Helada' con valor categórico
    df['Helada'] = np.where(df[shift_col_name] < 0, 1, 0)

    # Eliminar la columna cambiada
    del df[shift_col_name]

    # Entrenamos el modelo y lo evaluamos con la validación cruzada
    for fold, (train_index, test_index) in enumerate(skf.split(df, df['Helada'])):
        train_data = df.iloc[train_index]
        test_data = df.iloc[test_index]

        X_train = train_data.drop('Helada', axis=1)
        y_train = train_data['Helada']

        X_test = test_data.drop('Helada', axis=1)
        y_test = test_data['Helada']

        clf = MLPClassifier(hidden_layer_sizes=hidden_units, activation='relu', max_iter=1000, random_state=random_state, alpha=alpha)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        clf.fit(X_train_scaled, y_train)
        y_pred = clf.predict(X_test_scaled)

        # Calcular la matriz de confusión
        cm = confusion_matrix(y_test, y_pred)

        # Obtener los pesos (coeficientes) del modelo
        weights = clf.coefs_

        # Calcular pesos promedio por característica
        avg_weights = np.mean(weights[0], axis=1)

        # Almacenar resultados
        result_dict = {
            'Hour': i,
            'Fold': fold,
            'TP': cm[0][0],
            'FP': cm[0][1],
            'FN': cm[1][0],
            'TN': cm[1][1],
        }

        # Calcular métricas manuales
        accuracy, precision, recall, f1_score, specificity = calculate_metrics_for_row(result_dict)

        # Agregar resultados manuales al diccionario
        result_dict.update({
            'Accuracy': accuracy,
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1_score,
            'Specificity': specificity
        })

        # Agregar pesos promedio para cada atributo
        for j, col in enumerate(train_data.columns):
            if col != 'Helada':
                result_dict[f'{col}'] = round(avg_weights[j], 6)

        results.append(result_dict)

    # Eliminar la columna 'Helada' para la próxima iteración
    del df['Helada']

# Convertir la lista de resultados a un DataFrame
result_df = pd.DataFrame(results)

#print(result_df)
#print("FIN")

#Se guardan los resultados
result_df.to_csv('resultados_ann.csv', index=False)

Se calcula el promedio para cada variable

In [ ]:
import pandas as pd

df = pd.read_csv('resultados_ann.csv')

# Agrupar por 'Hour' y 'Fold' y calcular el promedio de cada característica
df_promedio_iteracion = df.groupby(['Hour']).mean().reset_index()

# Redondear los resultados a 6 cifras decimales
df_promedio_iteracion = df_promedio_iteracion.round(6)

# Imprimir el DataFrame con los promedios por iteración
print('Promedio de cada característica por iteración:')
print(df_promedio_iteracion)

# Guardar el DataFrame redondeado en un nuevo archivo CSV
df_promedio_iteracion.to_csv('resultados_ann_promedio.csv', index=False)

Graficamos cada ann

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Supongamos que tienes cuatro DataFrames df_8, df_16, df_32, df_64
# Ajusta los nombres de las variables según tus datos reales
df_8 = pd.read_csv('ann_03_8_plus_promedio.csv')
df_16 = pd.read_csv('ann_03_16_plus_promedio.csv')
df_32 = pd.read_csv('ann_03_32_plus_promedio.csv')
df_64 = pd.read_csv('ann_03_64_plus_promedio.csv')

dfs = [df_8, df_16, df_32, df_64]  # Lista de DataFrames
hidden_units = [8, 16, 32, 64]  # Lista de unidades ocultas correspondientes

metric_columns = df_8.columns[6:11]  # Ajusta según tus nombres reales de columnas

# Crear un gráfico de referencia de colores
reference_colors = plt.cm.get_cmap('tab10', len(dfs))

# Crear el gráfico de referencia
plt.figure(figsize=(8, 2))
for i, color in enumerate(reference_colors.colors):
    plt.plot([], [], color=color, label=f'Unidades ocultas: {hidden_units[i]}')

plt.title('Referencia de colores')
plt.axis('off')
plt.legend(loc='center')

# Guardar el gráfico de referencia
#plt.savefig('color_reference_metrics.png')

# Crear gráficos de líneas para cada métrica y cada DataFrame
plt.figure(figsize=(15, 10))  # Ajustar el tamaño según sea necesario

for i, metric in enumerate(metric_columns):
    ax = plt.subplot(4, 3, i+1)  # Cambiar a 4 filas y 3 columnas

    for j, df in enumerate(dfs):
        plt.plot(df['Hour'], df[metric], color=reference_colors(j))

    plt.title(f'{metric}')
    plt.xlabel('Hora')

    # Agregar cuadrícula (grid)
    plt.grid(True)

    # Mostrar todos los valores del eje x de 0 a 23
    plt.xticks(range(24))

    # Rotar los valores del eje x verticalmente y ajustar su tamaño de fuente
    plt.xticks(rotation='vertical')
    plt.gca().xaxis.set_tick_params(labelsize=8)

    # Obtener número de la imagen y agregarlo en la esquina superior derecha
    ax.text(0.95, 0.95, f'{i+1}', transform=ax.transAxes,
            verticalalignment='top', horizontalalignment='right',
            fontsize=10, color='black', fontweight='bold', bbox=dict(facecolor='white', alpha=1))

# Ajustar espacio entre subgráficos y entre el título y los subgráficos
plt.tight_layout(pad=2.0)

# Guardar el gráfico de comparación de métricas de la matriz de confusión
plt.savefig('confusion_matrix_metrics_comparison.png')

plt.show()